# 🫁 Pneumonia Detection with Severity Assessment

**Pipeline:** Dataset → Train ResNet-50 → Pneumonia Image → Inference + Severity → Grid Analysis → Grad-CAM → Final Report → Model Evaluation

| Module | Purpose |
|--------|--------|
| `config.py` | Centralized hyperparameters and paths |
| `data_loader.py` | Dataset loading, augmentation, train/val split |
| `model.py` | ResNet-50 build, train loop, checkpointing |
| `inference.py` | Preprocessing, prediction, severity scoring |
| `visualization.py` | Grid lines, Grad-CAM, severity gauge, reports |

---
## Step 0 — Install Dependencies

In [ ]:
!pip install -q torch torchvision tqdm pytorch-grad-cam opencv-python matplotlib scikit-learn seaborn

---
## Step 1 — Load the Binary Dataset (NORMAL vs PNEUMONIA)

The dataset has a binary folder structure:
```
chest_xray/
  train/
    NORMAL/      ← class 0
    PNEUMONIA/   ← class 1
  test/
    NORMAL/
    PNEUMONIA/
```
We create a **validation split** from the training set to monitor overfitting.

In [ ]:
from data_loader import load_datasets

train_loader, val_loader, test_loader, class_names = load_datasets()
print(f"\nClass mapping: {dict(enumerate(class_names))}")
print(f"  0 = {class_names[0]}")
print(f"  1 = {class_names[1]}")

---
## Step 2 — Train the Model

Fine-tunes a **ResNet-50** pretrained on ImageNet, adapted for:
- 1-channel grayscale input (pretrained weights averaged, not discarded)
- 2-class output (NORMAL vs PNEUMONIA)
- Includes validation accuracy per epoch + learning rate scheduling

In [ ]:
from model import build_model, train_model
from visualization import plot_training_history

model = build_model()
print(f"Model on: {next(model.parameters()).device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

history = train_model(model, train_loader, val_loader)
plot_training_history(history)

---
## Step 3 — Pick a Pneumonia Image → Inference & Severity Assessment

We take a **PNEUMONIA** image from the test set, run it through the trained model, and grade severity using pixel-level opacity analysis.

In [ ]:
import os, glob
from config import DATA_DIR
from inference import preprocess_image, predict, calculate_severity
from visualization import show_lung_roi

# --- Pick a Pneumonia image from the test set ---
pneumonia_dir = os.path.join(DATA_DIR, 'test', 'PNEUMONIA')
pneumonia_images = sorted(glob.glob(os.path.join(pneumonia_dir, '*')))
print(f"Found {len(pneumonia_images)} pneumonia test images")

# Select a specific image (change index to try others)
img_path = pneumonia_images[0]
print(f"Selected: {os.path.basename(img_path)}\n")

# --- Preprocess & Predict ---
tensor, original_img = preprocess_image(img_path)
pred_idx, diagnosis, confidence = predict(model, tensor)

# --- Severity Assessment ---
severity_label, score = calculate_severity(original_img, pred_idx)

print(f"  Diagnosis:  {diagnosis}")
print(f"  Confidence: {confidence:.2%}")
print(f"  Severity:   {severity_label}")
print(f"  Opacity:    {score:.4f}")

# Show original + lung mask
show_lung_roi(original_img)

---
## Step 4 — Grid-Line Region Analysis

Divides the X-ray into a **4×4 grid**, computes per-region opacity, and highlights:
- 🟢 **Green** = Clear lung tissue (<15% opacity)
- 🟠 **Orange** = Medium opacity (15–35%)
- 🔴 **Red** = Consolidated/infected region (>35%)

In [ ]:
from visualization import analyze_grid_regions

# 4x4 grid analysis (increase grid_size for finer analysis)
grid_scores = analyze_grid_regions(original_img, grid_size=4)

---
## Step 5 — Grad-CAM Visualization

Uses **Gradient-weighted Class Activation Mapping** on ResNet-50's `layer4` to show which regions the model focused on when making its prediction.

In [ ]:
from visualization import generate_gradcam, show_gradcam_panel

visualization, grayscale_cam = generate_gradcam(model, tensor, original_img)
show_gradcam_panel(original_img, visualization, grayscale_cam)

---
## Step 6 — Final Diagnostic Report

Combines all analyses into a single 4-panel report: Original → Grad-CAM → Severity Gauge → Text Summary.

In [ ]:
from visualization import show_final_report

show_final_report(
    original_img, visualization,
    severity_label, score,
    diagnosis, confidence,
    grid_scores=grid_scores
)

---
## Step 7 — Model Evaluation on Full Test Set

Runs the model on **all test images** and produces:
- Classification report (precision, recall, F1)
- Confusion matrix heatmap

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from config import DEVICE, CLASS_NAMES

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, preds = torch.max(model(inputs), 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification Report
print("\n" + "=" * 50)
print("  CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quick stats
accuracy = np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
print(f"\n\u2705 Overall Test Accuracy: {accuracy:.2%}")